In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import joblib # для збереження моделі

In [11]:
# Комірка 2-3 (Об'єднана)
categories = ["Season", "Time", "Weather"]
models = {} # Тут будемо зберігати навчені моделі

# Визначаємо колонки
numeric_features = ['BPM', 'Energy', 'Dance', 'Acoustic', 'Valence']
categorical_features = ['Key'] 

for cat in categories:
    print(f"\n🚀 Тренуємо модель для: {cat}")
    
    # Завантажуємо датасет
    df = pd.read_csv(f"{cat.lower()}.csv")
    
    # Видаляємо рядки з пропусками
    df = df.dropna(subset=numeric_features + categorical_features + [f'{cat}_Label'])
    
    X = df[numeric_features + categorical_features]
    y = df[f'{cat}_Label']
    
    # Розбиваємо на трен/тест
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # 🔥 ОСЬ ФІКС: Створюємо НОВИЙ препроцесор для КОЖНОЇ категорії окремо! 🔥
    numeric_transformer = StandardScaler()
    categorical_transformer = OneHotEncoder(handle_unknown='ignore')
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ])
    
    # Створюємо повний пайплайн
    clf = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42))
    ])
    
    # Навчаємо
    clf.fit(X_train, y_train)
    
    # Оцінюємо
    score = clf.score(X_test, y_test)
    print(f"📊 Точність ({cat}): {score:.2f}")
    
    # Зберігаємо готову незалежну модель
    models[cat] = clf


🚀 Тренуємо модель для: Season
📊 Точність (Season): 0.46

🚀 Тренуємо модель для: Time
📊 Точність (Time): 0.42

🚀 Тренуємо модель для: Weather
📊 Точність (Weather): 0.46


In [13]:
# Завантажуємо тестовий плейлист summer_test.csv
test_df = pd.read_csv('summer_test.csv')

# Вибираємо потрібні колонки для моделі
features = ['BPM', 'Energy', 'Dance', 'Acoustic', 'Valence', 'Key']

print(f"Створюємо портрет плейлиста з {len(test_df)} треків")
print("=" * 50)

# Обчислюємо середні значення для числових ознак
numeric_features = ['BPM', 'Energy', 'Dance', 'Acoustic', 'Valence']
playlist_profile = test_df[numeric_features].mean()

# Для Key вибираємо найпоширеніший
most_common_key = test_df['Key'].mode()[0]

# Створюємо профіль плейлиста
playlist_profile['Key'] = most_common_key

print("Портрет плейлиста:")
for feat in features:
    if feat in numeric_features:
        print(f"  - {feat}: {playlist_profile[feat]:.2f}")
    else:
        print(f"  - {feat}: {playlist_profile[feat]}")

print("\n" + "=" * 50)
print("Прогнози моделі для портрета плейлиста:")
print("=" * 50)

# Перетворюємо на DataFrame в 1 рядок
input_df = pd.DataFrame([playlist_profile])

for cat in categories:
    model = models[cat]
    
    # Отримуємо всі можливі класи для цієї моделі
    classes = model.classes_
    
    # Отримуємо ймовірності
    probabilities = model.predict_proba(input_df)[0]
    
    print(f"\n[{cat.upper()}] Профіль:")
    
    # Сортуємо результати від найбільшого до найменшого
    results = list(zip(classes, probabilities))
    results.sort(key=lambda x: x[1], reverse=True)
    
    for label, prob in results:
        print(f"  - {label}: {prob * 100:.1f}%")

Створюємо портрет плейлиста з 53 треків
Портрет плейлиста:
  - BPM: 122.94
  - Energy: 68.25
  - Dance: 69.42
  - Acoustic: 12.96
  - Valence: 51.11
  - Key: A Major

Прогнози моделі для портрета плейлиста:

[SEASON] Профіль:
  - summer: 38.7%
  - winter: 30.3%
  - spring: 18.0%
  - autumn: 13.0%

[TIME] Профіль:
  - morning: 49.0%
  - evening: 28.0%
  - night: 12.0%
  - afternoon: 11.0%

[WEATHER] Профіль:
  - sunny: 50.0%
  - cloudy: 20.0%
  - rainy: 19.0%
  - snowy: 11.0%
